AUDIO

In [ ]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/'

X_train =  np.load(BASE + 'X_train_v3.npy')
y_train =  np.load(BASE + 'y_train_v3.npy')
X_val   =  np.load(BASE + 'X_val_v3.npy')
y_val   =  np.load(BASE + 'y_val_v3.npy')
X_test  =  np.load(BASE + 'X_test_v3.npy')
y_test  =  np.load(BASE + 'y_test_v3.npy')


Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_cnn_gru(input_shape=(42, 130), n_classes=5):
    inputs = tf.keras.Input(shape=input_shape, name='audio_input')

    x = layers.Permute((2, 1))(inputs)          # (130, 42)

    # CNN block
    x = layers.Conv1D(128, 5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)               # (65, 128)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(256, 5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)               # (32, 256)
    x = layers.Dropout(0.3)(x)

    # GRU block
    x = layers.GRU(128, return_sequences=True)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.GRU(64)(x)
    x = layers.Dropout(0.3)(x)

    # this plugs into the face fusion later
    audio_vec = layers.Dense(128, activation='relu', name='audio_vec')(x)
    output    = layers.Dense(n_classes, activation='softmax', name='audio_output')(audio_vec)

    return models.Model(inputs=inputs, outputs=output)

In [ ]:
model = build_cnn_gru()
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=4, factor=0.5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nAccuracy: {acc:.2%}")
model.save(BASE + 'audio_cnn_gru_v3.keras')

FACE BEGIN

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from collections import Counter

from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# configurations / Paths
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTISM_PATH = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/FER-Autism/Autism emotion recogition dataset/Autism emotion recogition dataset/train'

FERAC_PATH = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/FERAC/FERAC Dataset/train'

BASE = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/'

os.makedirs(BASE, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16

In [ ]:
# ============================================================
# LABEL MAPS
# ============================================================
# We are using only 5-class:
# anger, fear, happy, neutral, sad
#
# FERAC has no sad class, so sad only comes from Talaat.
# although it can makes sad weaker/ or domain-specific.
# And Later because of this lack of Autism Data problem
# I would turn this PROJECT INTO CLASSIFYING WHETHER AN AUTISTIC CHILD IS HAPPY OR NOT
# ============================================================

AUTISM_MAP = {
    'anger':   'anger',
    'fear':    'fear',
    'joy':     'happy',
    'natural': 'neutral',
    'sadness': 'sad',
}

FERAC_MAP = {
    'anger':   'anger',
    'fear':    'fear',
    'joy':     'happy',
    'natural': 'neutral',

}

In [ ]:
# ============================================================
# IMAGE LOADING FUNCTION
# ============================================================

def load_images(root_path, label_map, dataset_name):
    images = []
    labels = []
    paths = []

    skipped_folders = []
    failed_files = []

    for folder in sorted(os.listdir(root_path)):
        folder_path = os.path.join(root_path, folder)

        if not os.path.isdir(folder_path):
            continue

        key = folder.lower().strip()

        if key not in label_map:
            skipped_folders.append(folder)
            continue

        standard_label = label_map[key]

        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            fpath = os.path.join(folder_path, fname)

            try:
                img = load_img(
                    fpath,
                    target_size=(IMG_SIZE, IMG_SIZE),
                    color_mode='rgb'
                )

                arr = img_to_array(img).astype('float32')  # keep 0-255

                images.append(arr)
                labels.append(standard_label)
                paths.append(fpath)

            except Exception as e:
                if len(failed_files) < 5:
                    failed_files.append((fpath, str(e)))

    print(f"\n{dataset_name}")
    print(f"Loaded images: {len(images)}")
    print(f"Distribution: {Counter(labels)}")

    if skipped_folders:
        print(f"Skipped folders: {skipped_folders}")

    if failed_files:
        print("Failed files:")
        for f, err in failed_files:
            print(f, err)

    return images, labels, paths

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

print("Loading Talaat FER-Autism dataset...")
a_imgs, a_labs, a_paths = load_images(
    AUTISM_PATH,
    AUTISM_MAP,
    "Talaat FER-Autism"
)

print("\nLoading FERAC dataset...")
f_imgs, f_labs, f_paths = load_images(
    FERAC_PATH,
    FERAC_MAP,
    "FERAC"
)

X_raw = np.array(a_imgs + f_imgs, dtype='float32')
y_str = np.array(a_labs + f_labs)
all_paths = np.array(a_paths + f_paths)

print("\nCombined shape:", X_raw.shape)
print("Combined distribution:", Counter(y_str))

Loading Talaat FER-Autism dataset...

Talaat FER-Autism
Loaded images: 999
Distribution: Counter({'neutral': 200, 'anger': 200, 'fear': 200, 'happy': 200, 'sad': 199})
Skipped folders: ['surprise']

Loading FERAC dataset...

FERAC
Loaded images: 615
Distribution: Counter({'happy': 370, 'neutral': 147, 'anger': 59, 'fear': 39})

Combined shape: (1614, 224, 224, 3)
Combined distribution: Counter({np.str_('happy'): 570, np.str_('neutral'): 347, np.str_('anger'): 259, np.str_('fear'): 239, np.str_('sad'): 199})


In [ ]:
# ============================================================
# ENCODE LABELS
# ============================================================

le_face = LabelEncoder()
y_enc = le_face.fit_transform(y_str)

print("\nLabel classes:", le_face.classes_)
print("Label encoding:", dict(zip(le_face.classes_, range(len(le_face.classes_)))))

NUM_CLASSES = len(le_face.classes_)
print("Number of classes:", NUM_CLASSES)


Label classes: ['anger' 'fear' 'happy' 'neutral' 'sad']
Label encoding: {np.str_('anger'): 0, np.str_('fear'): 1, np.str_('happy'): 2, np.str_('neutral'): 3, np.str_('sad'): 4}
Number of classes: 5


In [ ]:
# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

X_train_raw, X_val_raw, y_train, y_val, paths_train, paths_val = train_test_split(
    X_raw,
    y_enc,
    all_paths,
    test_size=0.2,
    random_state=SEED,
    stratify=y_enc
)

print("Train:", len(X_train_raw))
print("Val:", len(X_val_raw))

print("\nTrain distribution:", Counter(le_face.inverse_transform(y_train)))
print("Val distribution:", Counter(le_face.inverse_transform(y_val)))

Train: 1291
Val: 323

Train distribution: Counter({np.str_('happy'): 456, np.str_('neutral'): 278, np.str_('anger'): 207, np.str_('fear'): 191, np.str_('sad'): 159})
Val distribution: Counter({np.str_('happy'): 114, np.str_('neutral'): 69, np.str_('anger'): 52, np.str_('fear'): 48, np.str_('sad'): 40})


In [ ]:
# ============================================================
# MOBILENETV2 PREPROCESSING
# ============================================================
# MobileNetV2 expects images processed with:
# tf.keras.applications.mobilenet_v2.preprocess_input
#
# ============================================================

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

X_train = preprocess_input(X_train_raw.copy())
X_val = preprocess_input(X_val_raw.copy())

print("Train pixel range after preprocessing:", X_train.min(), X_train.max())
print("Val pixel range after preprocessing:", X_val.min(), X_val.max())

Train pixel range after preprocessing: -1.0 1.0
Val pixel range after preprocessing: -1.0 1.0


In [ ]:
# ============================================================
# CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(1.2473429951690822), 1: np.float64(1.3518324607329844), 2: np.float64(0.5662280701754386), 3: np.float64(0.9287769784172661), 4: np.float64(1.6238993710691825)}


In [ ]:
# ============================================================
# DATASET PIPELINE
# ============================================================

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

train_ds = (
    train_ds
    .shuffle(buffer_size=len(X_train), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# ============================================================
# DATA AUGMENTATION
# ============================================================
#
# ============================================================

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.04),
        layers.RandomZoom(0.08),
        layers.RandomTranslation(0.05, 0.05),
    ],
    name="data_augmentation"
)

In [ ]:
# ============================================================
# SIMPLE MOBILENETV2 MODEL
# ============================================================

base = applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="face_input")

x = data_augmentation(inputs)

x = base(x, training=False)

x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)

x = layers.Dense(128, activation='relu', name="dense_128")(x)
x = layers.Dropout(0.4, name="dropout")(x)

# This is the vector we would join with audio_vec later
face_vec = layers.Dense(128, activation='relu', name='face_vec')(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='face_output')(face_vec)

face_model = models.Model(inputs, outputs, name="mobilenetv2_face_emotion")

face_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "mobilenetv2_face_emotion"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ face_input (InputLayer)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pool                 │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ face_vec (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ face_output (Dense)             │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,439,109 (9.30 MB)

 Trainable params: 181,125 (707.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# ============================================================
# COMPILE PHASE 1
# ============================================================

face_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ============================================================
# CALLBACKS
# ============================================================

best_model_path = BASE + 'best_face_mobilenetv2.keras'

callbacks = [
    ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# ============================================================
# PHASE 1: TRAINING ONLY THE NEW TOP LAYERS
# ============================================================

print("\nPhase 1: Training top layers only...")

history1 = face_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


Phase 1: Training top layers only...
Epoch 1/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 809ms/step - accuracy: 0.2711 - loss: 1.6702
Epoch 1: val_accuracy improved from None to 0.28793, saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 89s 980ms/step - accuracy: 0.2866 - loss: 1.6326 - val_accuracy: 0.2879 - val_loss: 1.5420 - learning_rate: 0.0010
Epoch 2/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 763ms/step - accuracy: 0.3772 - loss: 1.4876
Epoch 2: val_accuracy improved from 0.28793 to 0.42724, saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 76s 905ms/step - accuracy: 0.3873 - loss: 1.4617 - val_accuracy: 0.4272 - val_loss

In [ ]:
# ============================================================
# PHASE 2: FINE-TUNING ONLY THE LAST PART OF MOBILENETV2
# ============================================================

base.trainable = True

# Freeze most layers.
# Am only fine-tuning the last 30 layers.
for layer in base.layers[:-30]:
    layer.trainable = False

# Keeping BatchNorm frozen.
# because this is important for small datasets.
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

face_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nPhase 2: Fine-tuning last MobileNetV2 layers...")

history2 = face_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


Phase 2: Fine-tuning last MobileNetV2 layers...
Epoch 1/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 657ms/step - accuracy: 0.2010 - loss: 1.6921
Epoch 1: val_accuracy improved from None to 0.20124, saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 71s 796ms/step - accuracy: 0.2246 - loss: 1.6548 - val_accuracy: 0.2012 - val_loss: 1.5974 - learning_rate: 1.0000e-05
Epoch 2/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 636ms/step - accuracy: 0.2574 - loss: 1.5902
Epoch 2: val_accuracy improved from 0.20124 to 0.29721, saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/best_face_mobilenetv2.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 62s 768ms/step - accuracy: 0.2835 - loss: 1.5606 - val_accuracy: 0.

In [ ]:
# ============================================================
# LOAD BEST MODEL AND EVALUATE
# ============================================================

face_model = tf.keras.models.load_model(best_model_path)

loss, acc = face_model.evaluate(val_ds, verbose=0)

print(f"\nBest validation accuracy: {acc:.2%}")
print(f"Validation loss: {loss:.4f}")


Best validation accuracy: 67.80%
Validation loss: 0.9122


In [ ]:
# Saving
np.save(BASE + 'X_face_train_raw.npy', X_train_raw)
np.save(BASE + 'y_face_train.npy',     y_train)
np.save(BASE + 'X_face_val_raw.npy',   X_val_raw)
np.save(BASE + 'y_face_val.npy',       y_val)

END FACE

In [ ]:
# ══════════════════════════════════════════════════════════════════
# COMBINED AUXILIARY LOSS MODEL
# Both branches train together. Face is the one teaching audio.
# At inference: only audio branch runs on the wearable.
# ══════════════════════════════════════════════════════════════════

import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.metrics import classification_report
from google.colab import drive

drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

BASE = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/'

# ── LOAD AUDIO DATA ───────────────────────────────────────────────
# These are the ZCR+RMS+MFCC features from CREMA-D + RAVDESS
X_audio_tr = np.load(BASE + 'X_train_v3.npy')
y_audio_tr = np.load(BASE + 'y_train_v3.npy')
X_audio_va = np.load(BASE + 'X_val_v3.npy')
y_audio_va = np.load(BASE + 'y_val_v3.npy')
X_audio_te = np.load(BASE + 'X_test_v3.npy')
y_audio_te = np.load(BASE + 'y_test_v3.npy')

print(f"Audio — train: {X_audio_tr.shape} | val: {X_audio_va.shape} | test: {X_audio_te.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Audio — train: (25326, 42, 130) | val: (2814, 42, 130) | test: (7035, 42, 130)


In [ ]:
# ── LOAD FACE DATA AND PREPROCESS ────────────────────────────────

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

X_face_tr_raw = np.load(BASE + 'X_face_train_raw.npy')
X_face_va_raw = np.load(BASE + 'X_face_val_raw.npy')
y_face_tr     = np.load(BASE + 'y_face_train.npy')
y_face_va     = np.load(BASE + 'y_face_val.npy')

X_face_tr = preprocess_input(X_face_tr_raw.copy())
X_face_va = preprocess_input(X_face_va_raw.copy())

print(f"Face  — train: {X_face_tr.shape} | val: {X_face_va.shape}")

Face  — train: (1291, 224, 224, 3) | val: (323, 224, 224, 3)


In [ ]:
# -------------- PAIR AUDIO WITH FACE BY EMOTION CLASS ----------------
# Each audio sample needs a face sample of the SAME emotion
# We randomly pick one - "synthetic pairing"
# it also changes every time this runs

def pair_by_class_indices(y_audio, y_face):
    by_class = {}
    for idx, lbl in enumerate(y_face):
        by_class.setdefault(int(lbl), []).append(idx)

    indices = []
    n_face = len(y_face)
    for lbl in y_audio:
        pool = by_class.get(int(lbl), [])
        indices.append(random.choice(pool) if pool else random.randint(0, n_face - 1))
    return np.array(indices, dtype=np.int32)

print("Computing face index mappings...")
face_idx_tr = pair_by_class_indices(y_audio_tr, y_face_tr)
face_idx_va = pair_by_class_indices(y_audio_va, y_face_va)
print(f"Index arrays: train={face_idx_tr.shape}, val={face_idx_va.shape}")

Computing face index mappings...
Index arrays: train=(25326,), val=(2814,)


In [ ]:
BATCH = 32

def make_dataset(X_audio, X_face_pool, face_idx, y_labels, shuffle=True):
    face_pool_tf = tf.constant(X_face_pool, dtype=tf.float32)
    audio_tf     = tf.constant(X_audio,     dtype=tf.float32)

    n = len(X_audio)
    ds = tf.data.Dataset.from_tensor_slices(
        (np.arange(n, dtype=np.int32), face_idx, y_labels)
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2048, seed=SEED)

    def lookup(audio_i, face_i, label):
        audio = audio_tf[audio_i]
        face  = face_pool_tf[face_i]
        return (
            {'audio_input': audio, 'face_input': face},
            {'audio_output': label, 'combined_output': label}
        )

    return ds.map(lookup, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_audio_tr, X_face_tr, face_idx_tr, y_audio_tr, shuffle=True)
val_ds   = make_dataset(X_audio_va, X_face_va, face_idx_va, y_audio_va, shuffle=False)

print("Datasets ready.")

Datasets ready.


In [ ]:
# ------------------ LOAD PRE-TRAINED MODELS -----------------------------
# Both models are already trained — we load their saved weights
print("\nLoading saved models...")
audio_model = tf.keras.models.load_model(BASE + 'audio_cnn_gru_v3.keras')
face_model  = tf.keras.models.load_model(BASE + 'best_face_mobilenetv2.keras')

# ----------------- EXTRACT FEATURE VECTOR SUB-MODELS --------------------------
# We cut each model at its 128-dim feature vector layer
# and throw away their original classification heads
# New heads are built fresh inside the combined model

audio_branch = tf.keras.Model(
    inputs  = audio_model.input,
    outputs = audio_model.get_layer('audio_vec').output,
    name    = 'audio_branch'
)

face_branch = tf.keras.Model(
    inputs  = face_model.input,
    outputs = face_model.get_layer('face_vec').output,
    name    = 'face_branch'
)

# Freeze face branch - it's the teacher, it doesn't need to change
# Only audio branch receives gradient updates
face_branch.trainable = False

print(f"Audio branch: {audio_branch.output_shape}")
print(f"Face branch:  {face_branch.output_shape}")



Loading saved models...
Audio branch: (None, 128)
Face branch:  (None, 128)


In [ ]:
#--------------------------- BUILD COMBINED MODEL ---------------------------------
audio_input = tf.keras.Input(shape=(42, 130),     name='audio_input')
face_input  = tf.keras.Input(shape=(224, 224, 3), name='face_input')

# Run both inputs through their branches
audio_vec = audio_branch(audio_input)   # (batch, 128)
face_vec  = face_branch(face_input)     # (batch, 128)

# ------------- HEAD A: audio-only output -----------------
# This is the INFERENCE head — what the wearable uses
# Gets its own loss so the audio branch stays independently capable
audio_output = layers.Dense(
    5, activation='softmax', name='audio_output'
)(audio_vec)

# ---------- FUSION: combining both feature vectors -------
# Concatenate audio(128) + face(128) → 256-dim combined vector
fused = layers.Concatenate(name='fusion')([audio_vec, face_vec])
fused = layers.Dense(128, activation='relu')(fused)
fused = layers.Dropout(0.3)(fused)

# ── HEAD B: combined output ───────────────────────────────────────
# Uses both audio + face signal
# pushes audio branch to align with autism face patterns
combined_output = layers.Dense(
    5, activation='softmax', name='combined_output'
)(fused)

combined_model = models.Model(
    inputs  = [audio_input, face_input],
    outputs = [audio_output, combined_output],
    name    = 'combined_model'
)

combined_model.summary()

Model: "combined_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ audio_input         │ (None, 42, 130)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ face_input          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_branch        │ (None, 128)       │    386,432 │ audio_input[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ face_branch         │ (None, 128)       │  2,438,464 │ face_input[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion              │ (None, 256)       │          0 │ audio_branch[0][… │
│ (Concatenate)       │                   │            │ face_branch[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     32,896 │ fusion[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_output        │ (None, 5)         │        645 │ audio_branch[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_output     │ (None, 5)         │        645 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,859,082 (10.91 MB)

 Trainable params: 419,850 (1.60 MB)

 Non-trainable params: 2,439,232 (9.30 MB)

In [ ]:
# ── COMPILE WITH AUXILIARY LOSS ───────────────────────────────────
# total_loss = 0.4 × audio_loss + 0.6 × combined_loss
# The 0.6 weight means face supervision is the stronger signal
combined_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss = {
        'audio_output':    'sparse_categorical_crossentropy',
        'combined_output': 'sparse_categorical_crossentropy'
    },
    loss_weights = {
        'audio_output':    0.4,
        'combined_output': 0.6
    },
    metrics = {
        'audio_output':    'accuracy',
        'combined_output': 'accuracy'
    }
)

In [ ]:
# -------------------- TRAIN---------------------------
callbacks = [
    EarlyStopping(
        monitor='val_audio_output_accuracy',
        patience=8,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_audio_output_accuracy',
        patience=4,
        factor=0.5,
        min_lr=1e-7,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        BASE + 'combined_model_best.keras',
        monitor='val_audio_output_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("\nTraining combined model...")
history = combined_model.fit(
    train_ds,
    validation_data = val_ds,
    epochs    = 3,
    callbacks = callbacks
)


Training combined model...
Epoch 1/3
792/792 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - audio_output_accuracy: 0.7954 - audio_output_loss: 0.7046 - combined_output_accuracy: 0.8651 - combined_output_loss: 0.4429 - loss: 0.5476
Epoch 1: val_audio_output_accuracy improved from None to 0.80526, saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/combined_model_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/USAII/Datasets/combined_model_best.keras
792/792 ━━━━━━━━━━━━━━━━━━━━ 1009s 1s/step - audio_output_accuracy: 0.8054 - audio_output_loss: 0.6638 - combined_output_accuracy: 0.8707 - combined_output_loss: 0.4158 - loss: 0.5150 - val_audio_output_accuracy: 0.8053 - val_audio_output_loss: 0.5853 - val_combined_output_accuracy: 0.8717 - val_combined_output_loss: 0.3591 - val_loss: 0.4496 - learning_rate: 1.0000e-05
Epoch 2/3
792/792 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - audio_output_accuracy: 0.8203 - audio_output_loss: 0.5699 - combined_output_acc

In [ ]:

# ----------------- BUILD INFERENCE MODEL (audio only) ------------------------
# This is what runs on the wearable
# Face branch is completely removed - audio input → emotion output
inference_model = tf.keras.Model(
    inputs  = audio_input,
    outputs = audio_output,
    name    = 'inference_model'
)

# --------- EVALUATE ON HELD-OUT TEST SET -------------
print("\n=== FINAL RESULTS — INFERENCE MODEL (audio only) ===")
y_pred = np.argmax(inference_model.predict(X_audio_te), axis=1)

CLASSES = ['anger', 'fear', 'happy', 'neutral', 'sad']
print(classification_report(y_audio_te, y_pred, target_names=CLASSES))

# -------------------- SAVE ---------------------------
inference_model.save(BASE + 'inference_audio_final.keras')
combined_model.save(BASE + 'combined_model_final.keras')

print("\nSaved:")
print("  inference_audio_final.keras  ← runs on the wearable")
print("  combined_model_final.keras   ← full training model")

In [ ]:
# ----------- My RECOVERY CELL ------------
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Colab Notebooks/USAII/Datasets/'

# Load test data
X_audio_te = np.load(BASE + 'X_test_v3.npy')
y_audio_te = np.load(BASE + 'y_test_v3.npy')
print(f"Test set loaded: {X_audio_te.shape}")

# Load the best combined model that was saved during training
combined_model = tf.keras.models.load_model(BASE + 'combined_model_best.keras')
print("Combined model loaded.")

# Extracting inference model - audio input → audio output only

inference_model = tf.keras.Model(
    inputs  = combined_model.input[0],   # audio_input is first
    outputs = combined_model.output[0],  # audio_output is first
    name    = 'inference_model'
)
print("Inference model extracted.")

# Evaluate on test set
print("\n=== FINAL RESULTS - INFERENCE MODEL (audio only) ===")
y_pred = np.argmax(inference_model.predict(X_audio_te), axis=1)

CLASSES = ['anger', 'fear', 'happy', 'neutral', 'sad']
print(classification_report(y_audio_te, y_pred, target_names=CLASSES))

# Save
inference_model.save(BASE + 'inference_audio_final.keras')
combined_model.save(BASE + 'combined_model_final.keras')

print("\nSaved:")
print("  inference_audio_final.keras  ← runs on the wearable")
print("  combined_model_final.keras   ← full training model")

Mounted at /content/drive
Test set loaded: (7035, 42, 130)
Combined model loaded.
Inference model extracted.

=== FINAL RESULTS — INFERENCE MODEL (audio only) ===
220/220 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step
              precision    recall  f1-score   support

       anger       0.96      0.87      0.91      1463
        fear       0.78      0.70      0.74      1463
       happy       0.89      0.74      0.81      1463
     neutral       0.76      0.84      0.80      1183
         sad       0.69      0.88      0.77      1463

    accuracy                           0.81      7035
   macro avg       0.82      0.81      0.81      7035
weighted avg       0.82      0.81      0.81      7035


Saved:
  inference_audio_final.keras  ← runs on the wearable
  combined_model_final.keras   ← full training model


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open(BASE + 'inference_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Size: {len(tflite_model)/1024:.1f} KB")
print("Done ✅")

Saved artifact at '/tmp/tmpgb5q4_5s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42, 130), dtype=tf.float32, name='audio_input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  137434254623376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254623952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254625680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254625872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254623760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254624912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254626448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254626640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254628368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254628560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137434254625296: Te

In [ ]:
# ONNX conversion
!pip install tf2onnx onnx -q

import tf2onnx, onnx

onnx_model, _ = tf2onnx.convert.from_keras(
    inference_model,
    opset=13,
    output_path=BASE + 'inference_model.onnx'
)

import os
size = os.path.getsize(BASE + 'inference_model.onnx')
print(f"Size: {size/1024:.1f} KB")
print("Done ✅ — download inference_model.onnx from Drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 27.7 MB/s eta 0:00:00
Size: 1542.5 KB
Done ✅ — download inference_model.onnx from Drive
